# 动态 Shape 执行流程 — RT2 与动静态混合调度

上一节我们看到：满足 Known Shape / 静态执行条件的整图或子图可以使用 Task Sink，执行期无需由 Host 逐算子 Launch。但现实中很多模型包含运行时才能确定 shape 的节点——例如 NLP 的变长序列、视觉的动态分辨率和数量不定的检测框。包含这些节点的完整图不能作为一个静态模型整体下沉；GE 会按条件划分 **Known Shape** 与 **Unknown Shape** 执行单元，前者仍可使用静态执行器，后者由动态执行路径处理。

本节聚焦 Unknown Shape 部分的运行时执行链路与开销来源，并介绍如何通过 profiling 区分 Host 与 Device 耗时，为后续第 4.5 节的动态 Shape 优化打基础。

本节学习大纲如下：

- 为什么 Unknown Shape 部分不能作为静态模型整体下沉
- 动静态混合执行边界
- 运行时处理链路：InferShape → Tiling → Buffer 获取/刷新 → Kernel Launch
- 动态 Shape 执行器（RT2.0）的执行循环
- 开销来源拆解
- 用 profiling 区分 Host 耗时与 Device 耗时
- 小结

## 1. 为什么 Unknown Shape 部分不能整体按静态图下沉

完整静态模型 Task Sink 要求编译期能够生成可预分发的任务序列：关键 Tensor 的 shape、内存布局以及不依赖运行时输入数据的 Tiling 等信息都必须可确定。Unknown Shape 节点打破了这一前提，其运行时 shape 相关工作需要进入 ExecuteGraph；但这并不意味着同一模型中的所有节点都必须退化为逐算子 Host 调度。

| 工作项 | Known Shape / 静态执行路径 | Unknown Shape 运行时路径 |
| --- | --- | --- |
| Shape 推导（InferShape） | 编译期使用已确定的输出 shape | 仅被判定为需要运行时 Shape 推导的节点在执行期推导 |
| Tiling 计算 | 不依赖运行时数据的 Tiling 可预计算；`tiling_depend` 需满足下沉条件 | 需要运行时 Tiling 的算子按真实 shape/数据计算；无 Tiling 算子除外 |
| 内存大小 / Buffer | 编译期完成模型级内存编排与复用 | 按真实 shape 获取或刷新输出/workspace Buffer，通常由缓存池复用 |
| 调度方式 | 满足条件的整图或 Known Shape 子图通过模型任务触发 | RT2 执行运行时节点；其中也可调用已下沉的 Known Shape 子图 |
| 可用编译优化 | 可充分利用融合、复用和 Task Sink | Unknown Shape 部分可用的静态优化通常较少 |

> 因此要区分“完整图”和“执行单元”：包含 Unknown Shape 节点的完整图不能作为一个静态模型整体 Task Sink，但满足规模和能力条件的 Known Shape 子图仍可走静态执行路径。相对同一实际 shape 的等价静态方案，Unknown Shape 路径通常会增加运行时 Host 工作；实际影响仍取决于图划分、算子计算量和缓存命中等因素。

## 2. RT2 运行时调度：执行期的处理链路

Unknown Shape 执行路径可以理解为：RT2 根据真实输入沿 ExecuteGraph 运行所需的 shape 推导、Tiling、Buffer 获取/刷新和 Kernel Launch 节点；若执行到 Known Shape 子图，则可直接触发其静态模型任务。下面展示的是一张动静态混合图中两类执行单元的典型路径，而不是要求每个原始算子都重复全部步骤：

<p align="center"><img src="./images/dynamic_host_dispatch.svg" alt="Unknown Shape 与 Known Shape 执行单元的处理链路" width="95%"></p>

### 2.1 InferShape：动态推导输出 shape

当节点被编译结果判定为需要运行时 Shape 推导时，RT2 才会在执行期根据真实输入运行 InferShape；Known Shape 节点可直接使用编译结果。GE 编译期与运行期复用同一套算子 InferShape 注册函数，以保持推导语义一致。

> 小知识：GE 内部对每个 Tensor 维护两套 shape——面向用户语义的 **OriginShape**（如 `[8,3,224,224]`）和面向硬件执行的 **StorageShape**（如 NC1HWC0 下的 `[8,1,224,224,16]`）。InferShape 先写 OriginShape，框架再换算出 StorageShape 用于内存与 Tiling。

### 2.2 Tiling 与内存分配：随 shape 而变

对需要运行时 Tiling 的 Unknown Shape AI Core 算子，shape 确定后才能计算 `block_dim`、`tiling_key`、workspace 大小等参数，并按真实大小获取或刷新输出/workspace Buffer。这里有两个边界：并非所有算子都有 Tiling；Known Shape 子图也不需要逐算子重复运行时 Tiling。另外，RT2 使用缓存内存分配器，执行期的“分配”通常是从内存池获取或复用 Buffer，不等同于每轮都向 Device 申请一块全新的物理内存。

## 3. 动态 Shape 执行器（RT2.0）的执行循环

GE 用专门的 **动态 Shape 执行器** 承载上述链路。新一代 RT2.0 执行器的核心设计是「**编译即执行准备**」：在编译期（Lowering 阶段）就把高层计算图 ComputeGraph 转换为可直接执行的 **ExecuteGraph**，运行时只需跑一个极简的节点循环，把「翻译」开销前移到编译期。

### 3.1 用户怎么走到动态 Shape 执行器

通常无需用户直接构造执行器。GE 会综合编译后的图属性、API/运行场景和配置选择执行路径：CANN 9.0 中包含 Unknown Shape 的普通图默认使用 RT2，但单算子场景、特定入口或 `ENABLE_RUNTIME_V2=0` 等配置可以改变路径；动态图内部还可能调用 Known Shape 静态子图。

<p align="center"><img src="./images/executor_selection.svg" alt="GE 根据图属性、入口和配置选择执行路径" width="90%"></p>

### 3.2 三子图生命周期

RT2.0 按模型生命周期把执行拆成三个子图阶段。Main Graph 是可重复执行的运行图，不仅包含实际计算，还可包含执行期 InferShape、Tiling、Buffer 获取/刷新和 Kernel Launch 等辅助节点：

| 阶段 | 子图 | 做什么 |
| --- | --- | --- |
| Load | Init Graph | 创建模型生命周期资源，完成流/分配器准备与常量初始化（执行一次后卸载）|
| Execute | Main Graph | 指定输入输出与运行参数，执行运行期辅助节点、静态子图调用和 Device 计算（可多次）|
| UnLoad | DeInit Graph | 清理并释放模型生命周期资源 |

> RT2.0 的顺序/拓扑执行器本身只负责调度通用节点函数；与运行时交互的动作由 Lowering 生成的注册节点表达。Init/DeInit 分离的是只需在加载/卸载阶段完成的资源生命周期工作，而每轮执行所需的动态 Buffer、Tiling 和 Launch 仍属于 Main Graph。

## 4. 开销来源拆解

相对同一实际输入 shape 的等价静态路径，Unknown Shape 路径通常会增加运行时的推导、备料与下发工作，但不应直接断言所有动态模型都更慢或一定是 Host Bound。需要结合实际图划分和 profiling 判断额外开销落在哪些节点。

| 开销项 | 产生位置 | 说明 |
| --- | --- | --- |
| InferShape 耗时 | Host | 对需要运行时 Shape 推导的节点按真实输入处理 |
| Tiling 耗时 | Host 或 AICPU | 对需要运行时 Tiling 的算子计算参数；无 Tiling/已静态化算子除外 |
| Buffer 获取/刷新 | Host | 按真实 shape 获取或刷新输出与 workspace，通常可复用内存池 |
| 运行时下发 | Host→Device | Unknown Shape 部分执行 Kernel Launch；Known Shape 子图可由一个模型任务触发 |
| Host-Device 同步 | Host↔Device | 部分场景需等待 Device 结果才能继续推导（数据依赖）|

当算子计算本身很轻、而上述 Host 工作很重时，模型会变成 **Host Bound**——Device 在等 Host 备料，timeline 上出现空泡。

```
 Host : |Infer|Tiling|Buffer|下发| |Infer|Tiling|Buffer|下发| ...
 Dev  :                  |算1|              ▯(空泡，等 Host) |算2| ...
          ↑ Unknown Shape 路径可能出现的 Host Bound 形态：Device 频繁等待 Host
```

> 若 profiling 确认瓶颈来自这些 Host 工作，可考虑减少运行时重复计算（如**动态分档**让命中档位的子图走静态优化）、让 AICPU 与 AI Core 并行，以及改善缓存/内存池复用。这些是第 4.5 节的主题。

### 4.1 动手实践：让同一动态图处理多种真实输入 Shape

下面构建输入描述为 `[-1, 512]` 的 `Relu → ReduceSum(axis=1)` 图，只向同一个 GE Session 添加一次，然后依次送入 batch 为 1、4、2 的 Tensor。每个 shape 先 warm-up 一次，再测量 5 次同步 `RunGraph` 调用，打印真实输入/输出 shape、P50/P95 端到端时间，并和 NumPy 对拍。

这样可以直接观察动态图在运行期接收不同 shape，而不是用固定公式模拟 InferShape、Tiling 或内存大小。示例显式设置 `precision_mode_v2=origin`，避免默认 FP16 路径在 512 个元素的 ReduceSum 累加中放大量化误差。若要拆分每轮 Host InferShape/Tiling 与 Device kernel 时间，需继续按第 5 节采集 profiling。

> **耗时提示**：首次 warm-up `RunGraph` 需要完成动态图编译，在 CANN 9.0 环境中可能需要数十秒或更长；计时结果不包含该次 warm-up。这里测得的是包含 Host 调度、Device 执行和结果返回的 `RunGraph` E2E，不是单纯的 NPU Kernel 时间；后续 batch=4/2 会复用同一张已编译图。


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)
# === 真机运行：同一 GE 动态图接受多种 batch，并在 NPU 上执行 ===
import time

import numpy as np
from ge.es.graph_builder import GraphBuilder
from ge.es.math import ReduceSum
from ge.es.nn import Relu
from ge.ge_global import GeApi
from ge.graph import Tensor
from ge.graph.types import DataType, Format
from ge.session import Session

DEVICE_ID = 0
GRAPH_ID = 1
FEATURES = 512
MEASURE_RUNS = 5

builder = GraphBuilder("DynamicReluReduceGraph")
x = builder.create_input(
    index=0,
    name="input_x",
    data_type=DataType.DT_FLOAT,
    shape=[-1, FEATURES],
)
relu = Relu(x)
reduced = ReduceSum(relu, [1], keep_dims=False)
builder.set_graph_output(reduced, 0)
graph = builder.build_and_reset()

ge_api = GeApi()
ge_api.ge_initialize({
    "ge.exec.deviceId": str(DEVICE_ID),
    "ge.graphRunMode": "0",
    "ge.exec.precision_mode_v2": "origin",
})
session = None
outputs = None
input_tensor = None
try:
    session = Session()
    print("[INFO] 正在 warm-up 并依次测量 batch=1/4/2", flush=True)
    session.add_graph(GRAPH_ID, graph)

    rng = np.random.default_rng(0)
    for batch in (1, 4, 2):
        input_array = rng.normal(size=(batch, FEATURES)).astype(np.float32)
        expected = np.maximum(input_array, 0).sum(axis=1)
        input_tensor = Tensor(
            input_array.reshape(-1).tolist(),
            None,
            DataType.DT_FLOAT,
            Format.FORMAT_ND,
            [batch, FEATURES],
        )

        # 当前 shape 先 warm-up 一次；batch=1 的首次调用同时完成编译/加载。
        outputs = session.run_graph(GRAPH_ID, [input_tensor])
        latencies_ms = []
        for _ in range(MEASURE_RUNS):
            started = time.perf_counter()
            outputs = session.run_graph(GRAPH_ID, [input_tensor])
            latencies_ms.append((time.perf_counter() - started) * 1e3)

        output_shape = list(outputs[0].shape)
        actual = np.asarray(outputs[0].data, dtype=np.float32).reshape(output_shape)
        np.testing.assert_allclose(actual, expected, rtol=1e-5, atol=1e-4)
        p50_ms, p95_ms = np.percentile(latencies_ms, [50, 95])
        print(
            "input_shape={} -> output_shape={}，RunGraph E2E P50/P95={:.3f}/{:.3f} ms（含 Host）".format(
                list(input_array.shape), output_shape, p50_ms, p95_ms
            )
        )

    print("[OK] 同一动态图已在 NPU 上依次处理 batch=1/4/2")
finally:
    outputs = None
    input_tensor = None
    # 释放 Session 引用，由 Session 析构统一释放图资源。
    session = None
    ge_api.ge_finalize()


## 5. 用 profiling 区分 Host 耗时与 Device 耗时

动态 Shape 调优的第一步是：**先搞清楚瓶颈在 Host 还是 Device**。GE Profiling 可分层观察 API、Host 运行时节点和 Device Task，帮助判断 Unknown Shape 路径的额外工作是否真的构成瓶颈。

### 5.1 采集命令与配置

```shell
# 离线 ACL 应用：msprof 拉起，重点开 task-time 看 Host/Device 时间线
msprof --application="./dyn_shape_infer" \
       --output=/tmp/prof_dyn \
       --runtime-api=on \
       --task-time=on \
       --aicpu=on \
       --aic-metrics=PipeUtilization
```

> `PipeUtilization` 描述单个 AI Core Task 内计算、Scalar、MTE 等流水单元的 cycle/耗时占比，不等同于整段执行期间的 Device 忙碌率。判断 Host Bound 应优先对齐 Host/Device timeline，并观察 Task 间空泡；需要采样型整体利用率时，应查看环境实际输出的 `ai_core_utilization` 等数据。

```cpp
// 在线 GeSession：通过 GE options 开启（profilingOptions 为 JSON）
std::map<ge::AscendString, ge::AscendString> config = {
    {"ge.exec.profilingMode",    "1"},
    {"ge.exec.profilingOptions", R"({"output":"/tmp/prof_dyn","task_trace":"on","aicpu":"on"})"}
};
ge::GEInitializeV2(config);
```

也可以用 C API 精确圈定稳定执行阶段，完整的配置与生命周期顺序为：`aclgrphProfInit → aclgrphProfCreateConfig → aclgrphProfStart → 执行模型 → aclgrphProfStop → aclgrphProfFinalize → aclgrphProfDestroyConfig`。其中 Create/Destroy 必须配对，避免把首次加载/编译耗时混入稳定阶段。

### 5.2 怎么读：Host Bound 还是 Device Bound

解析出 timeline 后，把同一次执行的 **Host 时间轴** 与 **Device 时间轴** 上下对齐看：

| 现象 | 判定 | 下一步 |
| --- | --- | --- |
| Host 线上运行期 InferShape/Tiling/Buffer 处理占比高，Device 线有大量空泡 | **Host Bound** | 走动态分档、减少 Host 重做、AICPU/AICore 并行 |
| Device 线连续繁忙、Host 线很短 | **Device Bound** | 瓶颈在算子本身，优化算子实现/精度，分档收益有限 |
| Host、Device 交替等待，频繁同步点 | **同步开销大** | 检查数据依赖、减少 Host-Device 往返 |

重点关注的几类条目：

- **Host 层**：Unknown Shape 节点的 InferShape、Tiling、Buffer 获取/刷新及 Launch 等相关耗时。
- **Device 层**：Task 执行时间和 Task 间空泡；`PipeUtilization` 用于分析 Task 内流水单元占比，而非整体忙碌率。
- **AICPU**：开 `--aicpu=on` 看 AI CPU 算子是否成为串行瓶颈。

```
读图模板（动态 shape 单次执行）：
  Host  : ▮Infer▮Tiling▮Buffer▮发▮  ......           ← 这段越长越可能是 Host Bound
  Device:                 ▮算子▮ ▯空泡▯ ▮算子▮ ...    ← 空泡多 = Device 在等 Host
```

> 经验法则：**动态 Shape 模型先量 Host/Device 占比，再决定优化方向**。盲目优化算子，遇到 Host Bound 时收益甚微。

## 6. 小结

- 包含 Unknown Shape 节点的完整图不能作为一个静态模型整体 Task Sink；满足条件的 Known Shape 子图仍可使用静态执行器。
- 对确需运行时处理的 Unknown Shape 节点，典型链路是 **InferShape → Tiling → Buffer 获取/刷新 → Kernel Launch**；已知 shape、无 Tiling 和静态子图路径属于例外。
- GE 用 RT2.0 承载动态执行路径，通过 Lowering 把 ComputeGraph 转换为 ExecuteGraph；Main Graph 同时包含运行时辅助节点、静态子图调用和实际计算。
- Unknown Shape 路径通常增加 Host 工作，但是否变成 Host Bound 取决于图划分、计算量、同步和缓存等实际情况。
- 调优第一步是用 **profiling 对齐 Host/Device timeline**；`PipeUtilization` 只反映 Task 内流水单元占比。

> 下一节 4.4 先介绍静态 Shape 执行优化；第 4.5 节再回到动态 Shape，通过**动态分档**、AICPU/AI Core 并行等手段减少经 profiling 确认的运行时开销。

## 课后练习

完成下列题目自测，如有错误建议结合本节对应小节复盘。

1. （判断题）包含 Unknown Shape 节点的完整图不能作为一个静态模型整体 Task Sink，但其中满足条件的 Known Shape 子图仍可能使用静态执行器。

2. （判断题）只要模型含有动态输入，模型内每个算子就都必须在每次执行时重新运行 InferShape 和 Tiling。

3. （判断题）GE 编译期和运行期复用同一套算子 InferShape 注册函数，以保证推导结果一致。

4. （单选题）对需要运行时 Shape/Tiling 的 Unknown Shape AI Core 节点，典型处理链路是？
    A. Tiling → InferShape → 下发 → 内存分配
    B. InferShape → Tiling → Buffer 获取/刷新 → 下发 Kernel
    C. 内存分配 → 下发 → InferShape → Tiling
    D. 下发 → InferShape → 内存分配 → Tiling

5. （单选题）RT2.0 动态执行器把模型执行拆成哪三个子图阶段？
    A. Prepare / Optimize / Build
    B. Init Graph / Main Graph / DeInit Graph
    C. Load / Save / Run
    D. Host / Device / Sync

6. （单选题）动态 Shape 模型 timeline 上 Host 线很长、Device 线大量空泡，应判定为？
    A. Device Bound，优先优化算子实现
    B. Host Bound，优先减少 Host 每次重做的工作（如动态分档）
    C. 内存不足
    D. 一切正常，无需优化

7. （多选题）以下关于 GE 动态 Shape 执行的描述，哪些是正确的？
    A. 仅被编译结果判定为需要运行时 Shape 推导的节点才在执行期运行 InferShape
    B. 动态/分区模型中的 Known Shape 子图仍可能由静态模型任务触发
    C. RT2 Main Graph 可以包含 Buffer 获取、Tiling、Kernel Launch 等运行时辅助节点
    D. PipeUtilization 等同于整段执行期间的 Device 忙碌率

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/04.03_answer.txt